# 02 — GLIMS selection to Parquet for Athena

**Purpose.** Turns notebook 01's selection into the Parquet file behind the Athena `glims_18k` table.
It adds `geog_area_rollup`, which buckets GLIMS's regional centres into the analysis regions (Asia,
North America, South America, Europe, Caucausus, Oceania), and drops the polygon geometry. The `sql/`
queries join on this table; the `identify_inference_*` queries use it to pick N glaciers per region.

## Inputs
- **`config.glims_selection_path()`**: the output of notebook 01.
- **Environment**: the project's uv environment as the kernel. No credentials needed.

## Outputs
- `data/glims_18k.parquet` (`config.GLIMS_PARQUET`, tracked).

## Manual step afterwards (not in code)
Upload it to `s3://training-images-metadata-t1-l2-sr/glims_18k/glims_18k.parquet`, which the Glue table
`glims_18k` in the `glacier-view` database (us-west-1) reads. Needs AWS write access; see
`.agents/references/aws.md` for bucket regions.

In [1]:
from glacierview import glims, log

log.configure()
glaciers = glims.add_region_rollup(glims.read_selection())
glaciers["geog_area_rollup"].value_counts(dropna=False)

geog_area_rollup
Asia             12625
North America     3704
South America      872
Europe             602
Caucausus          255
Oceania             35
Name: count, dtype: int64

In [2]:
glims.write_athena_table(glaciers);

2026-10-23 17:04:18,213 INFO    glacierview.glims | wrote 18093 glaciers to /Users/mattw/Desktop/projects/GlacierView/data/glims_18k.parquet
